# 00 - Data inventory and the clone manifest

Verify the deposit at the metadata level and build the one table every later
notebook reads: a row per (clone, sample) with its clone fraction.

Everything here runs on `scRNA/scRNA.metadata.csv` (19 MB). The expression
matrix is a **separate track** - see `scripts/convert_seurat_to_h5ad.R`. There
is no R in the Cowork VM, and the local disk is ~97% full, so that conversion
belongs on Greene.

**Confirms:** cell count, the family/passage design, clone-call rate per stage,
clone x stage presence, no cell assigned to two clones, and the clone
bottleneck.

**Blocks:** notebooks 01 and 02.

In [ ]:
import sys, pickle
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import etracer as et

P = et.Paths(Path.cwd().parent)
print(P)
print("metadata:", P.meta_csv.exists(), "| UMI rds:", P.umi_rds.exists())

In [ ]:
meta = et.load_metadata(P)
print(meta.shape)
meta.head(3)

In [ ]:
# --- the design, read off the sample names -------------------------------
# 3 BL tumours (one per founder family) -> 6 EE (two passages each) -> 6 LE.
# Each family therefore gives TWO observations of the same BL -> EE transition,
# which notebook 01 uses as within-family replication.
design = (meta.groupby(["stage", "family", "branch", "sample"])
              .size().rename("n_cells").reset_index()
              .sort_values(["stage", "family", "branch"]))
print(design.to_string(index=False))
assert len(design) == 15, "expected 15 samples"

In [ ]:
# --- clone-call rate per stage -------------------------------------------
# NOTE: earlier project notes recorded 49% / 91% / 95% for BL / EE / LE.
# The deposited metadata does not support that: the rate is roughly flat.
# Whichever number goes in the writeup has to come from this cell.
rate = (meta.groupby("stage")["called"].agg(called="sum", total="size")
            .assign(rate=lambda d: d["called"] / d["total"])
            .reindex(["BL", "EE", "LE"]))
print(rate.to_string())

In [ ]:
# --- leakage check: is any cell assigned to more than one clone? ----------
# This is the LineageFlow concern (connected components of the barcode-cell
# graph rather than nominal clone ids). Two things have to hold for
# "components == clones":
#   1. no static barcode is shared between clones
#   2. no cell carries barcodes from two clone groups
clone_bc = et.load_clone_barcodes(P)
all_bc = [b for v in clone_bc.values() for b in v]
shared = len(all_bc) - len(set(all_bc))
print(f"clones in pickle: {len(clone_bc)}")
print(f"static barcodes: {len(all_bc)} total, {len(set(all_bc))} unique -> {shared} shared")

lin = pd.read_csv(P.cell_lineage_csv)
per_cell = lin.groupby("cellBC")["lineageGrp"].nunique()
print(f"cells in lineage table: {len(per_cell)}, assigned to >1 clone: {(per_cell > 1).sum()}")

assert shared == 0, "a barcode is shared between clones - components != clones"
assert (per_cell > 1).sum() == 0, "a cell maps to >1 clone - group by components, not clone id"
print("\nOK: grouping by clone id is equivalent to grouping by components here.")

In [ ]:
# --- does the metadata agree with the standalone lineage table? -----------
lin_map = lin.drop_duplicates("cellBC").set_index("cellBC")["lineageGrp"]
j = meta.set_index("cell")["clone"].dropna()
both = j.index.intersection(lin_map.index)
print(f"called in metadata: {len(j)} | overlap with lineage table: {len(both)}")
print(f"disagreements: {(j.loc[both] != lin_map.loc[both]).sum()}")
print(f"in lineage table but not in scRNA metadata: {len(lin_map.index.difference(j.index))}"
      "   (expected: cells from the multiome / spatial assays)")

In [ ]:
# --- clone x stage presence ----------------------------------------------
pres = (meta.dropna(subset=["clone"])
            .groupby("clone")["stage"].apply(lambda s: frozenset(s)))
n_stages = pres.map(len).value_counts().sort_index()
all3 = pres[pres.map(len) == 3].index
fams = meta.dropna(subset=["clone"]).groupby("clone")["family"].nunique()

print(f"clones: {len(pres)}")
print(f"present at all three stages: {len(all3)}")
print("\nstages per clone:"); print(n_stages.to_string())
print("\nfamilies per clone:"); print(fams.value_counts().sort_index().to_string())
print(f"\nclones in all 3 families: {(fams == 3).sum()} "
      f"({(fams == 3).mean():.0%}) -- the cross-family sharing that makes a "
      "clone-identity lookup viable on a family hold-out")

In [ ]:
# --- nesting: do child tumours only contain clones from their parent? -----
bl = {f: set(d["clone"].dropna()) for f, d in meta[meta.stage == "BL"].groupby("family")}
for stage in ["EE", "LE"]:
    rows = []
    for (f, s), d in meta[meta.stage == stage].groupby(["family", "sample"]):
        got = set(d["clone"].dropna())
        rows.append((s, len(got), len(got - bl[f])))
    print(f"{stage}: " + ", ".join(f"{s} {n} clones ({o} not in parent BL)" for s, n, o in rows))

In [ ]:
# --- build and save the manifest -----------------------------------------
manifest = et.clone_manifest(meta)
summary = et.sample_summary(manifest)

manifest.to_csv(P.derived / "clone_manifest.csv", index=False)
summary.to_csv(P.derived / "sample_summary.csv", index=False)
print(f"manifest: {manifest.shape} -> {P.derived / 'clone_manifest.csv'}")
manifest.head()

In [ ]:
# --- the bottleneck ------------------------------------------------------
# Effective diversity collapsing from BL to EE/LE is why the honest n for this
# project is ~109 clones and not 91,411 cells.
print(summary[["sample", "stage", "family", "n_clones", "n_called",
               "top_clone_share", "eff_diversity_shannon",
               "median_cells_per_clone"]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
order = ["BL", "EE", "LE"]
for a, col, lab in zip(
    ax,
    ["eff_diversity_shannon", "top_clone_share", "median_cells_per_clone"],
    ["effective clone diversity\n(exp Shannon)", "share of the largest clone",
     "median cells per clone"],
):
    for i, st in enumerate(order):
        v = summary.loc[summary.stage == st, col]
        a.scatter([i] * len(v), v, s=42, alpha=0.85)
    a.set_xticks(range(3)); a.set_xticklabels(order)
    a.set_ylabel(lab); a.grid(alpha=0.25, axis="y")
ax[0].set_title("the bottleneck, per tumour")
plt.tight_layout(); plt.show()

In [ ]:
# --- cells per clone at BL: the two-stage sampling constraint -------------
# Distribution regression (Szabo et al. 2016) converges in BOTH the number of
# distributions and the samples per distribution. These are the second axis.
blm = manifest[manifest.stage == "BL"]
print(blm["n_cells"].describe().to_string())
for t in [5, 10, 20, 50, 100]:
    print(f"  clone-family units with >= {t:3d} BL cells: {(blm['n_cells'] >= t).sum():4d}")

## Findings

Fill in from the cells above before moving on:

- cells / clones / clones at all three stages:
- clone-call rate per stage (and whether the 49/91/95 note is retired):
- components == clones:
- nesting exact:
- effective diversity, BL -> EE:
- clone-family units with >= 20 BL cells:

Written to `derived/`: `clone_manifest.csv`, `sample_summary.csv`.